In [0]:
import subprocess
import sys
import re
import time
import os
import atexit
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pyspark import SparkConf
from pyspark import SparkContext
from pyspark import SQLContext
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import concat, col, udf, lag, date_add, explode, lit, unix_timestamp, regexp_extract, regexp_replace
from pyspark.sql.functions import month, weekofyear, dayofmonth, year, hour, minute, second, to_timestamp
from pyspark.sql.types import *
from pyspark.sql.types import DateType
from pyspark.sql.types import DataType
from pyspark.sql.window import Window
from pyspark.sql import Row
from pyspark.ml.classification import *
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler,OneHotEncoder,VectorIndexer, PCA, RFormula
from pyspark.ml import Pipeline, PipelineModel
from delta import DeltaTable

# Data Cleansing

In [0]:
df = spark.read.table("fraud_detection_project.bronze_layer.device_signals")

def StandardizeNames(df):
    l = df.columns
    cols = []
    for c in l:
        temp = re.sub(r'(?<!^)(?=[A-Z])', '_', c).lower()
        for _ in range(5):
            temp = re.sub(r'\b([a-z])_([a-z])\b', r'\1\2', temp)
            temp = re.sub(r'(?<=[a-z])_([a-z])(?=_|$)', r'\1', temp)
        temp = re.sub(r'_+','_', temp)
        temp = temp.lstrip('_')
        cols.append(temp)
    return df.toDF(*cols)
df = StandardizeNames(df)

df = df.withColumnRenamed("deviceid", "device_id")
df.dtypes

In [0]:
# Deleting duplicated data
df.dropDuplicates(['device_id'])

# Deleting rows without some features
df = df.dropna(how='any', subset=['device_id','file_path','ingest_datetime'])

# Feature Engineering